In [2]:
!unzip /content/mtd_dataset.zip -d /content/mtd_dataset


Archive:  /content/mtd_dataset.zip
   creating: /content/mtd_dataset/mtd_dataset/
   creating: /content/mtd_dataset/mtd_dataset/test/
   creating: /content/mtd_dataset/mtd_dataset/test/0/
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_0.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_1.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_10.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_11.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_12.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_13.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_14.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_15.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_16.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0class_sorted_17.jpg  
  inflating: /content/mtd_dataset/mtd_dataset/test/0/0

In [3]:
!find /content/mtd_dataset -maxdepth 3 -type d


/content/mtd_dataset
/content/mtd_dataset/mtd_dataset
/content/mtd_dataset/mtd_dataset/train
/content/mtd_dataset/mtd_dataset/train/4
/content/mtd_dataset/mtd_dataset/train/2
/content/mtd_dataset/mtd_dataset/train/good
/content/mtd_dataset/mtd_dataset/train/1
/content/mtd_dataset/mtd_dataset/train/3
/content/mtd_dataset/mtd_dataset/train/0
/content/mtd_dataset/mtd_dataset/test
/content/mtd_dataset/mtd_dataset/test/4
/content/mtd_dataset/mtd_dataset/test/2
/content/mtd_dataset/mtd_dataset/test/good
/content/mtd_dataset/mtd_dataset/test/1
/content/mtd_dataset/mtd_dataset/test/3
/content/mtd_dataset/mtd_dataset/test/0
/content/mtd_dataset/mtd_dataset/val
/content/mtd_dataset/mtd_dataset/val/4
/content/mtd_dataset/mtd_dataset/val/2
/content/mtd_dataset/mtd_dataset/val/good
/content/mtd_dataset/mtd_dataset/val/1
/content/mtd_dataset/mtd_dataset/val/3
/content/mtd_dataset/mtd_dataset/val/0


In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.utils import image_dataset_from_directory

print("GPU:", tf.config.list_physical_devices('GPU'))

BASE_PATH = "/content/mtd_dataset/mtd_dataset"
TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR   = os.path.join(BASE_PATH, "val")
TEST_DIR  = os.path.join(BASE_PATH, "test")

print("Train classes:", os.listdir(TRAIN_DIR))


GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Train classes: ['4', '2', 'good', '1', '3', '0']


In [2]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

test_ds = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

NUM_CLASSES = len(train_ds.class_names)
print("Detected classes:", train_ds.class_names)


Found 1144 files belonging to 6 classes.
Found 200 files belonging to 6 classes.
Found 200 files belonging to 6 classes.
Detected classes: ['0', '1', '2', '3', '4', 'good']


In [3]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # important for speed

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,726 (9.24 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [4]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 45s 777ms/step - accuracy: 0.6354 - loss: 1.4034 - val_accuracy: 0.5000 - val_loss: 1.8695
Epoch 2/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.7397 - loss: 0.9587 - val_accuracy: 0.5000 - val_loss: 1.5394
Epoch 3/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.7427 - loss: 0.9134 - val_accuracy: 0.5000 - val_loss: 1.5467
Epoch 4/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.7422 - loss: 0.8893 - val_accuracy: 0.5000 - val_loss: 1.5650
Epoch 5/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.7601 - loss: 0.8364 - val_accuracy: 0.5000 - val_loss: 1.6804


In [8]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2591 - loss: 1.9615
Test Accuracy: 0.5


In [10]:
model.save("wafer_model.keras")

In [11]:
!pip install -q tf2onnx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 18.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
ydf 0.14.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.3 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.


In [12]:
python -m tf2onnx.convert \
  --keras wafer_model.keras \
  --output wafer_model.onnx


SyntaxError: invalid syntax (ipython-input-3136015986.py, line 1)

In [13]:
!python -m tf2onnx.convert \
  --keras wafer_model.keras \
  --output wafer_model.onnx


2026-02-08 16:42:00.041280: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770568920.060981    4274 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770568920.066924    4274 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770568920.081736    4274 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568920.081768    4274 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770568920.081772    4274 computation_placer.cc:177] computation placer alr

In [14]:
model.export("wafer_savedmodel")


Saved artifact at 'wafer_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  137992124921808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992124922384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992124922576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992124922960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992124922000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992124922768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992123876368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992123877328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992123876944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137992123875408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1379921238

In [15]:
!python -m tf2onnx.convert \
  --saved-model wafer_savedmodel \
  --output wafer_model.onnx


2026-02-08 16:43:35.205642: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770569015.224932    4684 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770569015.230811    4684 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770569015.245896    4684 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770569015.245929    4684 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770569015.245933    4684 computation_placer.cc:177] computation placer alr

In [16]:
!pip install -q numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have n

In [5]:
model.export("wafer_savedmodel")


Saved artifact at 'wafer_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  137829901404688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901405648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901406608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901406032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901404880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901405456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901406416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901407952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901407568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137829901406224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1378299047

In [6]:
!python -m tf2onnx.convert \
  --saved-model wafer_savedmodel \
  --output wafer_model.onnx


2026-02-08 16:56:32.031274: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770569792.051134    8192 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770569792.057038    8192 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770569792.073309    8192 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770569792.073337    8192 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770569792.073341    8192 computation_placer.cc:177] computation placer alr

In [7]:
import os
os.listdir()


['.config',
 'wafer_model.keras',
 'mtd_dataset',
 'wafer_savedmodel',
 'mtd_dataset.zip',
 'wafer_model.onnx',
 'sample_data']

In [8]:
from google.colab import files
files.download("wafer_model.onnx")
files.download("mtd_dataset.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>